In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

missing_values_path = OUT_DIR / "features_derived.csv"

# Let pandas detect the delimiter
data = pd.read_csv(missing_values_path, sep=None, engine="python", encoding="utf-8-sig")
data

In [ ]:
data.columns

# Feature Standardization

In [ ]:
# =====================================================================
# MINIMAL STANDARDIZATION + VALIDATION PLAN  
# =====================================================================
# Goal:
#   Apply a general minimum basic treatment so the dataset is:
#     • safe for statistical analysis
#     • consistent in SI units / formatting
#     • free of weird numeric values (inf / huge magnitudes)
#     • correctly typed for later one-hot encoding (object for categoricals)
#   NOTE: No scaling/normalization here. No one-hot encoding here.
#
# Sections:
#   1) SI-Unit & Formatting Inspection (unique values)
#   2) Weird-Value Detection (inf / huge values)
#   3) Minimal Standardization:
#        3.1 Drop ID-like columns (keep SINH_ID)
#        3.2 Convert datetime columns to datetime64
#        3.3 Cast categorical columns to object dtype
#        3.4 Force binary/flag columns to Int64 (0/1)
#        3.5 Force continuous numeric columns to float
#        3.6 Replace ±inf with NaN
# =====================================================================


In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Examine correctness and formatting of numeric/coded values to ensure
# SI-unit consistency. Excludes ID, object, datetime, and binary columns.
# ---------------------------------------------------------------------

for col in data.columns:
    
    col_lower = col.lower()
    series = data[col]

    # Exclude ID columns
    if "id" in col_lower:
        continue
    
    # Exclude object/string columns
    if series.dtype == "object":
        continue
    
    # Exclude datetime columns
    if pd.api.types.is_datetime64_any_dtype(series):
        continue
    
    # Exclude binary columns
    unique_vals = series.dropna().unique()
    if len(unique_vals) <= 2 and set(unique_vals).issubset({0, 1, True, False}):
        continue
    
    print(f"Unique values in column: {col}")
    print(unique_vals)
    print("\n" + "="*80 + "\n")


Results: 

Looking at the results above, I do not see any garbage values, looks safe. 

In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Detect garbage values: ±inf and huge magnitudes. NaN/NaT excluded.
# ---------------------------------------------------------------------

weird_report = []

for col in data.columns:
    s = data[col]
    col_info = {"column": col}
    
    if pd.api.types.is_numeric_dtype(s):
        is_pos_inf = np.isposinf(s)
        is_neg_inf = np.isneginf(s)
        is_huge = (s.abs() > 1e12) & (~s.isna())

        col_info.update({
            "dtype": str(s.dtype),
            "pos_inf": int(is_pos_inf.sum()),
            "neg_inf": int(is_neg_inf.sum()),
            "huge(|x|>1e12)": int(is_huge.sum())
        })

        if col_info["pos_inf"] or col_info["neg_inf"] or col_info["huge(|x|>1e12)"]:
            weird_report.append(col_info)

    elif s.dtype == "object":
        s_str = s.astype(str).str.lower().str.strip()
        tok = {"inf", "+inf", "-inf", "infinity", "+infinity", "-infinity"}
        inf_str = s_str.isin(tok)
        if inf_str.any():
            col_info.update({
                "dtype": str(s.dtype),
                "string_inf": int(inf_str.sum())
            })
            weird_report.append(col_info)

if weird_report:
    pd.DataFrame(weird_report).sort_values("column")
else:
    print("No garbage values detected.")


In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Convert the exact known date columns present in dataset.
# ---------------------------------------------------------------------

date_cols = [
    'sdo5_degree_enrollment_date',
    'sdo1_previous_Final_Exam_Date',
    'sdo2_skc_SKC_DATUM',
    'sdo2_orientation_First_Event_Date',
    'sdo2_orientation_Last_Event_Date',
    'dropout_date'
]

# Show before
display(data[date_cols].head())

# Convert
for c in date_cols:
    data[c] = pd.to_datetime(data[c], errors='coerce')

# Show after
display(data[date_cols].head())
data[date_cols].dtypes

In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Convert the selected categorical columns to object dtype for later
# one-hot encoding, and then print their value counts (including NaN)
# for inspection of category distribution.
# ---------------------------------------------------------------------

categorical_cols = [
    'sdo5_degree_COLLEGEJAAR',
    'sdo5_degree_degree',
    'sdo1_previous_Previous_Education_Type',
    'sdo2_skc_ADVIES_DEF',
    'sdo2_orientation_Event_Types_Attended',
    'age_category'
]

# Cast to object dtype
for c in categorical_cols:
    data[c] = data[c].astype('object')

# Print value counts for inspection
for c in categorical_cols:
    print(f"\n=== Value counts for: {c} ===")
    print(data[c].value_counts(dropna=False))
    print("="*80)


In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# Confirm the above operation
# ---------------------------------------------------------------------
# Print the names of all object-type columns in `data` that contain
# more than two unique categories (including NaN as its own category).
# ---------------------------------------------------------------------

obj_cols = data.select_dtypes(include='object').columns

for col in obj_cols:
    unique_count = data[col].nunique(dropna=False)
    if unique_count > 2:
        print(col)

In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Replace gender codes:
#     M → 1
#     V → 0
# while leaving any unexpected or missing values unchanged.
# ---------------------------------------------------------------------

gender_map = {"M": 1, "V": 0}

data['sdo1_characteristics_gender'] = (
    data['sdo1_characteristics_gender']
    .map(gender_map)
    .astype('Int64')   # nullable integer type
)

# Optional: check result
data['sdo1_characteristics_gender'].value_counts(dropna=False)

In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Force all binary/flag fields to 0/1 integer (Int64) dtype.
# ---------------------------------------------------------------------

binary_cols = [
    'sdo5_degree_drop_out_with_propedeuse',
    'sdo5_degree_drop_out_without_propedeuse',
    'sdo5_degree_drop_out_to_other_degree_in_HU',
    'sdo5_degree_drop_out_temporary',
    'sdo5_degree_drop_out',
    'sdo5_degree_POSTAL_COUNTRY_NL',
    'sdo1_previous_Multiple_Previous_Education',
    'sdo6_results_Enrolled_for_at_least_one_exam',
    'has_previous', 'has_skc', 'has_orientation', 'has_dsa', 'has_nf',
    'sdo6_results_Average_Grade_A_Sat_Exam',
    'Multiple_Degrees_Enrollment',
    'sdo1_characteristics_Dutch_nationality',
    'sdo1_characteristics_gender'
    # missing flags
    'sdo2_orientation_Number_of_Event_Types_missing_flag',
    'sdo2_orientation_Number_of_Events_Attended_missing_flag',
    'sdo2_orientation_First_Event_Date_missing_flag',
    'sdo2_orientation_Last_Event_Date_missing_flag',
    'sdo1_previous_Final_Exam_Date_missing_flag',
    'sdo6_results_Percentage_Credits_A_missing_flag',
    'sdo6_results_Potential_Credits_A_missing_flag',
    'sdo1_prev_distance_distance_to_3584_CS_missing_flag',
    'sdo1_postal_distance_distance_to_3584_CS_missing_flag',
    'sdo1_characteristics_Dutch_nationality_missing_flag',
    'sdo2_skc_SKC_DATUM_missing_flag',
    'dropout_date_missing_flag',
    'time_first_orient_to_start_weeks_missing_flag',
    'time_last_orient_to_start_weeks_missing_flag',
    'gap_skc_to_start_weeks_missing_flag'
]

for c in binary_cols:
    data[c] = pd.to_numeric(data[c], errors='coerce').astype('Int64')


In [ ]:
# ---------------------------------------------------------------------
# PURPOSE
# ---------------------------------------------------------------------
# Convert continuous numeric measurement columns to float.
# ---------------------------------------------------------------------

numeric_float_cols = [
    'sdo2_orientation_Number_of_Events_Attended',
    'sdo2_orientation_Number_of_Event_Types',
    'sdo6_results_Total_Credits_Block_A',
    'sdo6_results_Potential_Credits_A',
    'sdo6_results_Percentage_Credits_A',
    'sdo6_results_Average_Grade_A',
    'sdo1_prev_distance_distance_to_3584_CS',
    'sdo1_postal_distance_distance_to_3584_CS',
    'change_in_distance',
    'time_first_orient_to_start_weeks',
    'time_last_orient_to_start_weeks',
    'gap_skc_to_start_weeks',
    'time_enroll_to_start_weeks',
    'gap_prev_exam_to_start_weeks'
]

for c in numeric_float_cols:
    data[c] = pd.to_numeric(data[c], errors='coerce').astype(float)


In [ ]:
# Save dataset
output_path = OUT_DIR / "features_standardized.csv"
data.to_csv(output_path, index=False)

print(f"✅ Successfully saved features_standardized.csv")
print(f"   Path: {output_path}")
print(f"   Shape: {data.shape}")
print(f"   Columns: {len(data.columns)}")